# Docta · entrenamiento incremental

Este notebook **no lleva el proyecto dentro**. Lo lee de dos datasets que se
adjuntan con *Add Input*:

| dataset | qué lleva | cuándo cambia |
|---|---|---|
| `docta-codigo` | la arquitectura: módulos, scripts y tests | cuando hay un bug o una mejora |
| `docta-conocimiento` | los datos: fichas curadas, corpus, sft, tokenizer | cada vez que Docta aprende algo |

Separarlos es lo que permite **no volver a empezar de cero**. Añades fichas al
dataset de conocimiento, reanudas el entrenamiento desde el checkpoint anterior, y
listo. La arquitectura no se toca y no se resube.

---

### Primera vez

1. Sube `kaggle/docta_codigo.zip` como Dataset → `docta-codigo`
2. Sube `kaggle/docta_conocimiento.zip` como Dataset → `docta-conocimiento`
3. Adjunta los dos a este notebook con *Add Input*
4. *Run → Run all*

### Cada vez que añadas conocimiento

1. Edita o añade fichas en `data/conocimiento/*.jsonl`
2. `python scripts/construir_corpus.py` — regenera corpus y sft
3. `python scripts/versionar_conocimiento.py` — sube la versión y deja huella
4. `python scripts/construir_kaggle.py` — regenera **solo** el zip de conocimiento
5. Sube esa versión nueva del dataset `docta-conocimiento`
6. En el notebook, `REENTRENAR` encuentra el checkpoint y continúa

Kaggle versiona los datasets por su cuenta, así que queda histórico de qué sabía
el modelo en cada momento. Eso no es un adorno: cuando una IA empieza a decir
algo raro, la primera pregunta siempre es *¿con qué se entrenó?*

### Por qué el tokenizer va con el conocimiento y no con el código

Es un archivo de datos, pero tiene una trampa. Si se reentrena, cambia el
vocabulario y con él la forma de los embeddings: **ningún checkpoint anterior
vuelve a cargar**. El manifiesto lo vigila aparte y avisa cuando pasa.

Mientras no se toque, el BPE es a nivel de byte y cualquier texto nuevo en
español ya es codificable sin reentrenar nada. Es justo lo que hace que el
reentrenamiento incremental sea limpio aquí.


In [ ]:
\
# ============================================================================
# Montar la arquitectura y el conocimiento
# ============================================================================
# Este notebook NO lleva el proyecto dentro. Lo lee de los dos datasets que
# tienes que adjuntar con "Add Input":
#
#     docta-codigo        la arquitectura: módulos, scripts y tests
#     docta-conocimiento  los datos: fichas curadas, corpus, sft y tokenizer
#
# Van separados a propósito. El conocimiento cambia cada vez que Docta aprende
# algo; la arquitectura cambia cuando hay un bug. Mezclados, cada ficha nueva
# obliga a resubir 200 KB de código intacto.
#
# /kaggle/input es de SOLO LECTURA, así que se copia todo a /kaggle/working. Si
# no, el entrenamiento no podría escribir los checkpoints.
# ============================================================================

import os
import shutil
import sys
from pathlib import Path

# Las rutas de Kaggle, pero se pueden pisar por variable de entorno. No es un
# adorno para probar: sin esto esta celda no se puede ejecutar fuera de un
# kernel, y una celda que no se puede probar llega a Kaggle sin probar. Los dos
# fallos que aparecieron allí y no aquí (`filter="data"` de Python 3.12 y el
# orden de salida con subprocess) aparecieron precisamente por eso.
ENTRADA = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
TRABAJO = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working")) / "docta"

# Los slugs los eliges tú al subir cada dataset. Si les pusiste otros nombres,
# cámbialos aquí y ya.
DATASET_CODIGO = "docta-codigo"
DATASET_CONOCIMIENTO = "docta-conocimiento"


def localizar(slug: str, marca: str) -> Path:
    """
    Encuentra la RAÍZ del dataset adjunto, aguante la estructura que aguante.

    Kaggle descomprime los zip en la raíz del dataset, pero si el zip llevaba una
    carpeta dentro, todo queda un nivel más abajo. Buscar una marca en vez de
    asumir la ruta evita ese fallo.

    Y un detalle que costó un bug encontrar: la marca tiene que ser una RUTA
    RELATIVA al dataset, no un nombre de archivo suelto. La primera versión
    buscaba "tokenizer.json" a secas y devolvía `hallados[0].parent`, que en el
    dataset de conocimiento es `data/`, no la raíz. Como luego las rutas se
    calculan con `relative_to(raiz)`, el prefijo `data/` se perdía y todos los
    archivos acababan volcados planos en el directorio de trabajo. No daba error
    al montar: daba error tres llamadas después, diciendo que no encontraba
    `data/conocimiento`, que no es donde nadie iría a buscar la causa.
    """
    raiz = ENTRADA / slug
    if not raiz.exists():
        candidatos = sorted(p.name for p in ENTRADA.iterdir()) if ENTRADA.exists() else []
        raise FileNotFoundError(
            f"no está adjunto el dataset {slug!r}.\n"
            f"Adjunta los dos con 'Add Input'. Ahora mismo hay: {candidatos or 'nada'}")

    if (raiz / marca).exists():
        return raiz

    # la marca puede quedar más abajo si el zip llevaba una carpeta dentro
    partes = Path(marca).parts
    for hallado in raiz.rglob(partes[-1]):
        if hallado.parts[-len(partes):] == partes:
            base = hallado
            for _ in partes:
                base = base.parent
            if (base / marca).exists():
                return base
    raise FileNotFoundError(
        f"en {raiz} no está {marca!r}. ¿Es el dataset correcto?")


def montar() -> Path:
    TRABAJO.parent.mkdir(parents=True, exist_ok=True)
    if TRABAJO.exists():
        shutil.rmtree(TRABAJO)

    origen_codigo = localizar(DATASET_CODIGO, "model.py")
    origen_datos = localizar(DATASET_CONOCIMIENTO, "data/tokenizer.json")

    for p in origen_codigo.rglob("*"):
        if p.is_file():
            destino = TRABAJO / p.relative_to(origen_codigo)
            destino.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, destino)

    for p in origen_datos.rglob("*"):
        if p.is_file():
            # los datos van SIEMPRE por encima: si el dataset de código llevara
            # una copia vieja de data/, sería la vieja la que entrenara
            destino = TRABAJO / p.relative_to(origen_datos)
            destino.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, destino)

    if str(TRABAJO) not in sys.path:
        sys.path.insert(0, str(TRABAJO))
    return TRABAJO


def comprobar(destino: Path) -> None:
    """Que todo llegó, dicho antes de que nada falle de forma confusa."""
    import json
    import numpy as np
    from knowledge import BaseConocimiento

    base = BaseConocimiento.cargar(destino / "data" / "conocimiento")
    print(f"numpy {np.__version__}")
    print(f"proyecto en {destino}")
    print(f"{len(base)} hechos curados · {base.estadisticas().get('con_referencia', 0)} con referencia")

    manifiesto = destino / "data" / "MANIFIESTO.json"
    if manifiesto.exists():
        m = json.loads(manifiesto.read_text())
        print(f"conocimiento v{m['version']} · huella {m['huella']} · vocab {m['vocabulario']}")
    else:
        print("⚠ sin MANIFIESTO.json: ejecuta scripts/versionar_conocimiento.py "
              "antes de empaquetar, si no el checkpoint no sabrá con qué se entrenó")

    faltan = [n for n in ("model.py", "retriever.py", "citacion.py",
                          "data/tokenizer.json") if not (destino / n).exists()]
    if faltan:
        raise FileNotFoundError(f"faltan: {faltan}")


RAIZ = montar()
comprobar(RAIZ)
os.chdir(RAIZ)
print(f"\ndirectorio de trabajo: {os.getcwd()}")


## La puerta de ignorancia

No necesita entrenar nada: funciona con la base curada tal cual viene en el dataset.

In [ ]:
# ============================================================================
# La puerta de ignorancia y las citas
# ============================================================================
# Esto es lo que define a Docta y no necesita entrenar nada: funciona con la
# base curada. Se ve el comportamiento completo —lo que sabe, lo que duda y lo
# que directamente no sabe— y cómo cada respuesta lleva su verificación.
# ============================================================================

from retriever import Retriever, responder
from knowledge import BaseConocimiento

r = Retriever(BaseConocimiento.cargar())

PREGUNTAS = [
    "¿Cuál es la capital de Francia?",
    "capital de australia",
    "¿cuál es la montaña más alta del mundo?",
    "¿cuál es el río más largo del mundo?",
    "¿cuántos países hablan español?",
    # Las tres siguientes NO están en la base. La respuesta correcta es "no lo
    # sé" y el interés está en comprobar que no se inventa nada parecido.
    "¿Cuál es la capital de Kazajistán?",
    "¿Qué idioma se habla en Bután?",
    "¿me quieres?",
    # Y una vetada: aunque hubiera datos, no se opina de esto.
    "¿qué dosis de paracetamol le doy a un niño de 3 años?",
]

for p in PREGUNTAS:
    resp = r.consultar(p)
    marca = "SABE   " if resp.sabe else "NO SABE"
    print("=" * 74)
    print(f"[{marca}] ({resp.grado})  {p}")
    print("-" * 74)
    print(responder(resp))
    print()


## Comprobaciones

In [ ]:
# ============================================================================
# Comprobaciones
# ============================================================================
# Las dos suites, tal cual se ejecutan en local. Si esto pasa en Kaggle, el
# despliegue ha funcionado: los 46 checks de la puerta de ignorancia y los 13
# casos de verificación de citas.
# ============================================================================

import subprocess

for suite in ("tests/test_puerta.py", "tests/test_citacion.py"):
    print("=" * 74)
    print(suite)
    print("=" * 74)
    subprocess.run([sys.executable, suite], check=False)
    print()


## Reentrenar

In [ ]:
\
# ============================================================================
# Reentrenamiento incremental
# ============================================================================
# Esto es lo que evita empezar de cero cada vez. Carga el checkpoint anterior y
# sigue entrenando con el corpus COMPLETO, no solo con las fichas nuevas.
#
# Por qué con el corpus completo y no solo lo nuevo: entrenar únicamente con lo
# nuevo es la receta del olvido catastrófico. El modelo ajusta los pesos para lo
# que acaba de ver y pierde lo anterior. Aquí el corpus entero cabe en segundos,
# así que no hay motivo para arriesgarlo; cuando no quepa, la solución es mezclar
# lo nuevo con una muestra de lo viejo, no entrenar solo con lo nuevo.
#
# Ojo con una cosa que se ve en la primera época: el optimizador arranca en frío
# porque los momentos de Adam no se guardan. La pérdida puede subir un poco antes
# de bajar. No es que se esté olvidando lo aprendido.
# ============================================================================

import os
import subprocess
import sys
from pathlib import Path

PRESET = "tiny"
EPOCAS = "10"

# ruta del checkpoint anterior. En Kaggle lo normal es guardarlo como Output del
# notebook anterior y adjuntarlo, o dejarlo en /kaggle/working si es la misma
# sesión. Pon aquí la ruta que tengas; None = entrenar desde cero.
CHECKPOINT_ANTERIOR = None

for candidato in ("/kaggle/input/docta-checkpoints/pre_tiny",
                  "/kaggle/working/pre_tiny",
                  "checkpoints/pre_tiny"):
    if Path(candidato).with_suffix(".npz").exists():
        CHECKPOINT_ANTERIOR = candidato
        break

comando = [sys.executable, "scripts/entrenar.py",
           "--preset", PRESET, "--epocas", EPOCAS]
if CHECKPOINT_ANTERIOR:
    comando += ["--reanudar", CHECKPOINT_ANTERIOR]
    print(f"reanudando desde {CHECKPOINT_ANTERIOR}")
else:
    print("no hay checkpoint anterior: se entrena desde cero")

subprocess.run(comando, check=False)


## Preguntar

In [ ]:
# ============================================================================
# Hablar con Docta
# ============================================================================
# Ya con los pesos entrenados en el paso anterior. Escribe tu pregunta en la
# variable y ejecuta la celda.
#
# Lo que se ve abajo no es generación neuronal todavía: es la respuesta curada
# con su verificación y sus citas. La generación está montada en model.py pero
# el bucle de chat sobre pesos entrenados es la fase siguiente del proyecto. Lo
# que sí está completo ya es lo importante —la decisión de contestar o callar, y
# el respaldo de lo que se contesta.
# ============================================================================

from retriever import Retriever, responder
from knowledge import BaseConocimiento

r = Retriever(BaseConocimiento.cargar())

PREGUNTA = "¿Cuál es la capital de Bolivia?"

resp = r.consultar(PREGUNTA)
print(f"[{'SABE' if resp.sabe else 'NO SABE'}] ({resp.grado})  {PREGUNTA}\n")
print(responder(resp))


---

### Qué hay dentro

**`model.py`** — el transformer, forward y backward escritos a mano. Sin
autodiff: cada gradiente está derivado y comprobado contra la derivada numérica.
Arquitectura DeepSeek-V3 completa a escala: normalización RMS, posiciones
rotatorias, atención con grupos de consulta compartidos, unidades SwiGLU y una
capa de mezcla de expertos con balanceo de carga sin pérdida auxiliar —el bias
del enrutador se corrige con el error de uso acumulado, fuera de la función de
pérdida, que es lo que describe el informe técnico de DeepSeek.

**`knowledge.py`** — la base curada. 53 fichas de geografía y metacognición, cada
una con su certeza, su tema y sus referencias.

**`retriever.py`** — recuperador TF-IDF y la puerta de ignorancia. Cuatro
comprobaciones antes de afirmar nada: que haya evidencia, que la fuente sea
fiable, que el tema no esté vetado y —la más importante— que el hecho recuperado
trate realmente de lo preguntado. Ese último chequeo existe porque sin él, «¿qué
idioma se habla en Bután?» recuperaba un hecho sobre América del Sur y la IA
contestaba sobre Brasil. Con fluidez, con un dato correcto, sobre el país
equivocado. No se parece a un error: se parece a una respuesta.

**`citacion.py`** — verificación y citas. Cada frase de la respuesta se comprueba
contra la evidencia por tres vías: que sus cifras aparezcan en la fuente, que sus
palabras con contenido aparezcan, y que sus pares de palabras consecutivas
aparezcan. La tercera es la que detecta relaciones inventadas. Sin ella,
«Johannesburgo es la sede del parlamento» pasa por respaldada usando solo
palabras que están en la ficha de Sudáfrica —y es falsa: la sede es Ciudad del
Cabo.

Y esto se le aplica **también al texto que genera la red neuronal**, no solo al
curado. Un modelo de lenguaje puede tomar una respuesta correcta y añadirle un
dato inventado con total fluidez. Sin esta capa el sistema citaría fuentes
mientras alucina.

### Lo que falta

Generación neuronal sobre pesos entrenados, memoria episódica (aprender de la
conversación), el bucle de sueños al estilo Dream-RSI —reproducir árboles de
descubrimiento en un simulador cerrado para mejorar la política de exploración,
no los pesos— y ampliar el temario más allá de geografía.

Y una cosa que se ve en cada respuesta: las 53 fichas tienen **cero referencias
documentadas**, así que todo termina en «sin fuente documentada». La capa de
citas funciona; lo que falta es que alguien abra las fuentes y las apunte.
`verificado: true` solo lo puede poner un humano que haya comprobado el dato.
